System construction and test


In [2]:
from datetime import date, datetime
import pandas as pd
#import yfinance as yf
import time
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'
from itertools import product
import sqlite3
from metrics import *


input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)


#define back testing
sqtitulos() , tbtitulo, symbol, intervalo, dataini, datafim

sqbacktesting
trading system , parameters , parameters range (max , min) , parameters steps
stoploss, stpl parameters, stpl parameters range , stpl parameters steps
stopdrawdown , stdr parameters(drawmax).max.min.step
index, indx parameters (comissions) 


In [5]:
id_backtest = 1

In [7]:
import sqlite3
import pandas as pd

# Conecta ao banco de dados SQLite
con = sqlite3.connect('sqTradeSys.db')  # ou o caminho correto do seu arquivo .sqlite

# Consulta SQL para extrair o registro
query = f"SELECT * FROM vwbacktest WHERE id_backtest = {id_backtest}"

# Executa a consulta e lê em um DataFrame
df = pd.read_sql_query(query, con)

# Converte o primeiro (e único) registro em Series
srbacktest = df.iloc[0] if not df.empty else None

# Fecha a conexão (opcional)
con.close()
display (srbacktest)

id_backtest                                                    1
id_titulos                                                     1
dataini                                      2020-04-03 19:30:00
datafim                                      2024-04-03 19:30:00
symbol                                                      GGAL
intervalo                                                  60min
moeda                                                        USD
source         C:\Users\scitr\anaconda_projects\Trading_Syste...
Name: 0, dtype: object

In [10]:
dataini = srbacktest['dataini']
datafim =  srbacktest['datafim']
display (dataini , datafim)

'2020-04-03 19:30:00'

'2024-04-03 19:30:00'

In [12]:
#%%timeit
import sqlite3
import pandas as pd
# Caminho para o banco de dados
caminho_bd =  srbacktest['source'] 
# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)
# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}' AND symbol = '{srbacktest['symbol']}' AND intervalo = '{srbacktest['intervalo']}' AND moeda = '{srbacktest['moeda']}'
ORDER BY datetime
"""
dftitulosdados = pd.read_sql_query(consulta, conexao)
# Fechando a conexão
conexao.close()
# Exibindo os primeiros registros para conferir
#display(dftitulosdados)
dftitulosdados = dftitulosdados.drop(columns=["symbol", "moeda", "intervalo"])
#display(len(dftitulosdados))
display(dftitulosdados.head(10))

,datetime,open,high,low,close,volume
0,2020-04-06 08:00:00,5.8034,5.8034,5.8034,5.8034,100.0
1,2020-04-06 09:00:00,6.0432,6.2510,5.9952,6.0832,114582.0
2,2020-04-06 10:00:00,6.0945,6.0945,5.6635,5.7954,159266.0
3,2020-04-06 11:00:00,5.7954,5.8034,5.4837,5.6835,242263.0
4,2020-04-06 12:00:00,5.6715,5.8354,5.6355,5.8274,186504.0
5,2020-04-06 13:00:00,5.8274,5.8354,5.7474,5.7834,68220.0
6,2020-04-06 14:00:00,5.7714,5.7794,5.6435,5.6435,84043.0
7,2020-04-06 15:00:00,5.6595,5.6595,5.5316,5.5956,122920.0
8,2020-04-06 16:00:00,5.6036,5.6036,5.6036,5.6036,49293.0
9,2020-04-07 09:00:00,5.9793,6.0672,5.7395,6.0032,101621.0


In [14]:
# Conecta ao banco de dados SQLite
con = sqlite3.connect('sqTradeSys.db')  # ou o caminho correto do seu arquivo .sqlite

# Consulta SQL para extrair o registro
#query = f"SELECT * FROM vwbacktestparameters ,name, max, min,step WHERE id_backtest = {id_backtest}"
query = f"SELECT  name, type, max, min, step FROM vwbacktestparameters WHERE id_backtest = {id_backtest}"
# Executa a consulta e lê em um DataFrame
df = pd.read_sql_query(query, con)

# Fecha a conexão (opcional)
con.close()

display (df)

,name,type,max,min,step
0,K,int,20,10,2
1,D,int,15,5,3
2,smoth,int,12,4,3
3,medM,int,10,4,2
4,lowM,int,10,4,2
5,stpl,float,0.04,0.01,0.01
6,comission,float,0.006,0.000,0.002
7,drawmax,float,0.2,0.1,0.1


In [16]:
import pandas as pd
import numpy as np
from itertools import product

def parameters_combinator(dfcomb: pd.DataFrame) -> pd.DataFrame:
    """
    Gera um DataFrame com todas as combinações possíveis de parâmetros
    definidos em dfcomb, respeitando os tipos especificados.

    Parâmetros esperados em dfcomb:
    - name: nome da coluna
    - type: tipo de dado ('int' ou 'float')
    - min: valor mínimo
    - max: valor máximo
    - step: incremento

    Retorna:
    - dfparamtest: DataFrame com todas as combinações possíveis
    """
    param_ranges = {}

    for _, row in dfcomb.iterrows():
        name = row['name']
        tipo = row['type']

        # Converte min, max, step para o tipo correto
        if tipo == 'int':
            min_val = int(row['min'])
            max_val = int(row['max'])
            step_val = int(row['step'])
        elif tipo == 'float':
            min_val = float(row['min'])
            max_val = float(row['max'])
            step_val = float(row['step'])
        else:
            raise ValueError(f"Tipo não suportado: {tipo}")

        # Gera a faixa de valores
        values = np.round(np.arange(min_val, max_val + step_val, step_val), 5)
        param_ranges[name] = values

    # Gera todas as combinações possíveis
    combinations = list(product(*param_ranges.values()))

    # Cria o novo DataFrame
    dfparamtest = pd.DataFrame(combinations, columns=param_ranges.keys())

    # Aplica os tipos definidos
    for _, row in dfcomb.iterrows():
        col = row['name']
        tipo = row['type']
        if tipo == 'int':
            dfparamtest[col] = dfparamtest[col].astype(int)
        elif tipo == 'float':
            dfparamtest[col] = dfparamtest[col].astype(float)

    return dfparamtest
dfcomb = df

In [18]:
dfcomb = df
dfparamtest = parameters_combinator(dfcomb)
print(dfparamtest.head())
print(len(dfparamtest))

    K  D  smoth  medM  lowM  stpl  comission  drawmax
0  10  5      4     4     4  0.01      0.000      0.1
1  10  5      4     4     4  0.01      0.000      0.2
2  10  5      4     4     4  0.01      0.000      0.3
3  10  5      4     4     4  0.01      0.002      0.1
4  10  5      4     4     4  0.01      0.002      0.2
92160


In [20]:
dfmetricas = None

START LOOP

In [23]:
il = 70000

In [25]:
# Parameters

# System parameters
# stoch_hml_1
K = dfparamtest.loc[il,'K']
D = dfparamtest.loc[il,'D']
smoth = dfparamtest.loc[il,'smoth']
medM = (dfparamtest.loc[il,'medM'])
lowM = (dfparamtest.loc[il,'lowM'])

# Backtesting parameters

stpl = dfparamtest.loc[il,'stpl']
comission = dfparamtest.loc[il,'comission']
drawmax = dfparamtest.loc[il,'drawmax']
print(K,D, smoth, stpl, drawmax)

18 11 13 0.02 0.2


In [27]:
#
lsmetricas = []
srparamtest = dfparamtest.loc[il]
lsmetricas.append(srparamtest)
#print(srparamtest)


Trading System calculation

In [30]:
from Stoch_HighMedLow_Long import *
dfsignals = Stoch_HighMedLow_Long (dftitulosdados, K, D, smoth, medM , lowM)
#display (dfsignals.head())


Stop Loss Reentry

In [33]:
from stoploss import *
dfsignals, dfstoploss = stop_loss_reentry (dfsignals, stpl)
#display (dfsignals.head())
#display (dfstoploss.head())


In [35]:
from index import *
dfindex = index_calculation (dfsignals, comission)
display (dfindex.head())


,datetime,open,high,low,close,volume,state,index_sc,trade,index
0,2020-12-01 18:00:00,7.2376,7.2376,7.2376,7.2376,100.0,enter,100.000000,0.000000,99.800000
1,2020-12-08 09:00:00,7.2780,7.3266,7.2214,7.2376,21139.0,out,100.000000,0.000000,99.800000
2,2020-12-29 11:00:00,7.2133,7.4074,7.2133,7.3993,226749.0,enter,100.000000,0.000000,99.800000
3,2020-12-30 13:00:00,7.3063,7.3185,7.2376,7.2484,87356.0,out,97.960618,-0.020394,97.764697
4,2021-01-20 13:00:00,6.3966,6.4046,6.3400,6.3965,51857.0,enter,97.960618,0.000000,97.764697


In [37]:
from stoploss import *
dfindexdrawdown = stop_drawdown_simple(dfindex, drawmax)
#display(dfindexdrawdown)

In [39]:
# METRICS
# creo dataframe para calculo de metricas
dfinputmetricas = dfindex[['datetime', 'state','index_sc', 'index','trade']]
#Metricas list inicialicion 

pd.set_option('display.max_rows', None)

#display(dfinputmetricas, lsmetricas)

In [41]:
from metrics import *
setirtotalanual , lsmetricas = tir_total_anualizada (dfinputmetricas, lsmetricas)
#display (setirtotalanual, lsmetricas)    

In [43]:
from metrics import *
dftiranual = tir_anuais_df (dfinputmetricas, dataini, datafim)
setiranuaisestats , lsmetricas = tir_anuais_estats(dftiranual, lsmetricas)
#display (dftiranual, lsmetricas)


In [45]:
from metrics import *
setradesestats, lsmetricas = trades_estats(dfinputmetricas, lsmetricas)
#display (lsmetricas)

In [ ]:

dfdrawdowns = drawdowns_df(dfinputmetricas)
sedrawdownsestats , lsmetricas = drawdowns_estats(dfdrawdowns, lsmetricas)
#display(lsmetricas)

In [ ]:

dfdiasout = dias_out_df (dfinputmetricas)
sediasoutestats, lsmetricas = dias_out_estats(dfdiasout, lsmetricas)
#display(lsmetricas)

In [ ]:
dfstopdrawdown = stop_drawdown_df (dfindexdrawdown)
sestopdrawdownestats, lsmetricas = stop_drawdown_estats(dfstopdrawdown, lsmetricas)
display (lsmetricas )

In [ ]:


def atualizar_df_metricas(dfmetricas, lsmetricas):
    """
    Adiciona uma linha ao DataFrame dfmetricas com os valores de lsmetricas.
    Se dfmetricas for None, cria o DataFrame com a estrutura das métricas.

    Parâmetros:
    - dfmetricas: pd.DataFrame ou None
    - lsmetricas: list de pd.Series

    Retorna:
    - pd.DataFrame atualizado
    """

    linha = pd.concat(lsmetricas)  # Une todas as Series em uma só

    if dfmetricas is None:
        # Cria o DataFrame com uma única linha
        dfmetricas = pd.DataFrame([linha.values], columns=linha.index)
    else:
        # Adiciona nova linha ao DataFrame existente
        dfmetricas.loc[len(dfmetricas)] = linha.values

    return dfmetricas

dfmetricas = atualizar_df_metricas(dfmetricas, lsmetricas)

In [ ]:
display (dfmetricas)

END LOOP

In [ ]:
dfmetricas.insert(
    loc=0,  # insere como primeira coluna
    column="id_backtest",
    value=[srbacktest["id_backtest"]] * len(dfmetricas)
)
display(dfmetricas)